# Production Road Metrics Colab

This notebook uses verified pretrained models for vehicle detection, outdoor metric-depth estimation, and road-scene segmentation. It validates the T4 runtime and model loading before processing video.

Metric values are labelled as measured, estimated, or unavailable. No lane, junction, speed, distance, or OpenDRIVE value is fabricated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/vamos-sujal/lane-opendrive.git'
REPO_DIR = Path('/content/lane-opendrive')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', Path.cwd())

In [ ]:
import subprocess
import sys

packages = [
    'numpy', 'scipy', 'opencv-python', 'Pillow', 'PyYAML',
    'transformers', 'accelerate', 'safetensors', 'ultralytics',
    'gdown', 'lxml', 'shapely', 'networkx',
]
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    '--only-binary=:all:',
] + packages, check=True)
print('Binary-wheel dependencies installed. Existing CUDA PyTorch was not replaced.')

In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'CUDA is required. Select a Colab T4 runtime.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('PyTorch:', torch.__version__)

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/lane-opendrive')
UFLD_DIR = Path('/content/Ultra-Fast-Lane-Detection')
if UFLD_DIR.exists():
    import shutil
    shutil.rmtree(UFLD_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/cfzd/Ultra-Fast-Lane-Detection.git',
    str(UFLD_DIR),
], check=True)

MODEL_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_config = __import__('yaml').safe_load(
    Path('/content/lane-opendrive/configs/model.yaml').read_text()
)
LANE_CHECKPOINT = MODEL_DIR / Path(model_config['checkpoint']['path']).name
if not LANE_CHECKPOINT.exists():
    subprocess.run([
        sys.executable, '/content/lane-opendrive/scripts/download_weights.py',
        '--config', '/content/lane-opendrive/configs/model.yaml',
        '--output-dir', str(MODEL_DIR),
    ], cwd=str(REPO_DIR), check=True)
subprocess.run([
    sys.executable, '/content/lane-opendrive/scripts/smoke_test.py',
    '--device', 'cuda',
    '--checkpoint', str(LANE_CHECKPOINT),
    '--repo-path', str(UFLD_DIR),
], cwd=str(REPO_DIR), check=True)
subprocess.run([
    sys.executable,
    '/content/lane-opendrive/scripts/verify_production_runtime.py',
    '--device', 'cuda',
], cwd=str(REPO_DIR), check=True)
print('Vehicle, depth, scene, and lane model checkpoints loaded successfully.')

In [ ]:
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/input')
RUNS_DIR = Path('/content/drive/MyDrive/lane_to_opendrive/production_runs')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

videos = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {'.mp4', '.mov', '.avi'}
)
if not videos:
    uploaded = files.upload()
    for name, content in uploaded.items():
        (INPUT_DIR / Path(name).name).write_bytes(content)
    videos = sorted(
        path for path in INPUT_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in {'.mp4', '.mov', '.avi'}
    )
if not videos:
    raise FileNotFoundError('No supported video found.')
if len(videos) > 1 and not (INPUT_DIR / 'input.mp4').exists():
    raise RuntimeError('Multiple videos found. Keep one or rename the intended video to input.mp4.')
VIDEO_PATH = str(INPUT_DIR)
print('Input:', VIDEO_PATH)

In [ ]:
import datetime
import subprocess
import sys

run_id = datetime.datetime.now(datetime.timezone.utc).strftime('run_%Y%m%dT%H%M%SZ')
RUN_DIR = RUNS_DIR / run_id
subprocess.run([
    sys.executable,
    '/content/lane-opendrive/scripts/run_production.py',
    '--input', VIDEO_PATH,
    '--output', str(RUN_DIR),
    '--device', 'cuda',
    '--scene-stride', '5',
    '--lane-checkpoint', str(LANE_CHECKPOINT),
    '--lane-repo-path', '/content/Ultra-Fast-Lane-Detection',
], cwd='/content/lane-opendrive', check=True)
print('Run directory:', RUN_DIR)

In [ ]:
import json

report = json.loads((RUN_DIR / 'production_report.json').read_text())
print(json.dumps({
    'input_path': report['input_path'],
    'frame_count': report['frame_count'],
    'fps': report['fps'],
    'duration_s': report['duration_s'],
    'processing_fps': report.get('processing_fps'),
    'calibration_status': report['calibration_status'],
    'scale_source': report['scale_source'],
    'dashcam_speed_mps': report.get('dashcam_speed_mps'),
    'dashcam_speed_kmh': report.get('dashcam_speed_kmh'),
    'dashcam_speed_status': report.get('dashcam_speed_status'),
    'distance_travelled_m': report.get('distance_travelled_m'),
    'distance_travelled_status': report.get('distance_travelled_status'),
    'road_length_m': report.get('road_length_m'),
    'road_length_status': report.get('road_length_status'),
    'lane_validation': report.get('lane_validation'),
    'lane_graph': report.get('lane_graph'),
    'opendrive': report.get('opendrive'),
    'capabilities': report['capabilities'],
    'models': report['models'],
}, indent=2))
print('Full report:', RUN_DIR / 'production_report.json')